# Deploy and Monitor — `amygda_ops_risk_score`

## What this notebook does

Continues from `02_risk_score_guide.ipynb`. Takes the **complete model zip** made by
`generate_risk_scores()` (Step 12) and keeps scoring new data with it, with no session
and no retraining. Haven't run the first two stages? A ready-made model trained on the
sample data is used instead (`examples/sample_data/models/`).

1. **Deploy** the model once. A small *scoring bundle* is stored on Amygda's servers.
2. **Score** new logs whenever they arrive: a one-off file, or every day or week.
3. **Monitor**: the `amygda-risk` command (or `Monitor` in Python) keeps a folder on your
   machine with the history, the scores, a record of each run and a dashboard, and
   updates them every time new data arrives.

| Section | What you do |
|---|---|
| 1 | Decide whether to deploy, and see what is stored |
| 2 | Build the scoring bundle and look inside it |
| 3 | Deploy |
| 4 | Score a one-off file |
| 5 | Why history matters (`lookback` and `score_from`) |
| 6 | Set up a monitor |
| 7 | Run it on weekly batches |
| 8 | Schedule it |
| 9 | Run health, and when to retrain |
| 10 | Manage and delete |

The examples use the sample data: a fixed-log model trained on 2023, and the logs of
Jan–Mar 2024 split into 13 weekly files (`sample_data/batches/`). The same steps work
for free-text models; see `03b_deploy_and_monitor_free_text.ipynb`.

### Every time you open this notebook — run these cells first

In [ ]:
import glob, json, os, shutil, sys, webbrowser
from pathlib import Path

import pandas as pd

# Installing the SDK first time:
#   !pip install git+https://github.com/Amygda/amygda_ops_risk_score_sdk.git
# Already installed, please update to the latest version:
#   !pip install --upgrade git+https://github.com/Amygda/amygda_ops_risk_score_sdk.git

import amygda_ops_risk_score
from amygda_ops_risk_score import OpsRiskClient, helpers
from amygda_ops_risk_score.monitor import Monitor, MonitorError

# The `amygda-risk` command. If this notebook's kernel can't find it on PATH (its
# environment isn't activated), the same command is run through this kernel's Python.
AMYGDA_RISK = shutil.which("amygda-risk") or f'"{sys.executable}" -m amygda_ops_risk_score.monitor.cli'

print("amygda_ops_risk_score version:", amygda_ops_risk_score.__version__)

# How to get your API key

The Amygda API portal is coming soon!

Once available, you will be able to:
- Log in at https://portal.amygda.io (COMING SOON)
- Go to API Keys → Create Key, copy your key, paste it below.

In the meantime, to request an API key contact the Amygda CEO directly:
  faizan@amygdalabs.com

In [ ]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

API_KEY = os.environ.get("AMYGDA_SDK_API_KEY") or "paste_your_api_key_here"   # ← paste your key here

# The monitor and the `amygda-risk` command read the key from AMYGDA_API_KEY.
# It is never written to the monitor folder.
os.environ["AMYGDA_API_KEY"] = API_KEY

client = OpsRiskClient()
client.wait_until_ready()
print("API is ready.")

In [ ]:
# ── Paths — edit once ────────────────────────────────────────────────────────
# The complete model zip from Step 12: yours from examples 02a/02b if you ran them,
# otherwise the ready-made sample model (fixed-log, trained on the 2023 sample data).
MODEL_ZIPS = sorted(glob.glob("../examples/outputs/fixed_log/complete_model_*.zip"), key=os.path.getmtime)
MODEL      = MODEL_ZIPS[-1] if MODEL_ZIPS else "../examples/sample_data/models/fixed_log_complete_model.zip"

SAMPLES    = Path("../examples/sample_data")
TRAINING   = SAMPLES / "fixed_log_training.csv"        # the logs the model was trained on
BATCHES    = SAMPLES / "batches" / "fixed_log"         # week_01.csv … week_13.csv (Jan–Mar 2024)

DEPLOYMENT_NAME = "guide-fleet"                        # unique within your organisation
MON_DIR    = Path("outputs/guide_fleet_monitor")       # the monitor folder (section 6)
MON_NAME   = "guide-fleet-monitor"

week = {int(p.stem[-2:]): p for p in sorted(BATCHES.glob("week_*.csv"))}
print("Model:", MODEL)
print(f"{len(week)} weekly files in {BATCHES}")

In [ ]:
# Start clean: remove what an earlier run of this notebook left behind
for d in client.list_deployments(API_KEY):
    if d["name"] in (DEPLOYMENT_NAME, MON_NAME):
        client.delete_deployment(API_KEY, d["deployment_id"])
        print("Deleted earlier deployment", d["name"])
shutil.rmtree(MON_DIR, ignore_errors=True)

---
## 1. When to deploy, and what is stored

Deploy once you are happy with a risk score run: the scores after Step 12 make sense, the
checks passed and the alert bands are agreed. The deployed model then scores new logs
exactly as Step 12 would have, so you don't need a session for every new file.

**What deploying stores.** Only a small *scoring bundle*, built on your machine from the
model zip:

| Stored on Amygda's servers | Never uploaded |
|---|---|
| Settings (columns, frequency, rolling window) | Your log rows |
| System and subsystem names, weights and calibration thresholds | Asset IDs and timestamps |
| Log codes (fixed-log) or keywords and their vectors (free-text) | Failures and scores |

It is kept until you delete it, or until it has been unused for 90 days. When you score,
the logs you send are used for that call only and are not kept.

**If you can't accept that,** keep scoring through sessions: import the complete model zip
in a new session and go straight to Step 11. A self-hosted scorer, which runs on your own
machines, is coming soon.

The exact wording, which you must accept explicitly when you deploy:

In [ ]:
print(client.deployment_storage_policy())

---
## 2. Build the scoring bundle and look inside it

`deploy_model` builds the bundle for you. Building it here first lets you see exactly
what would be uploaded.

**What you should see:** a few small JSON files and the log-code map, a few kB to a few
MB, against a model zip that also holds your training and scored data.

In [ ]:
bundle = helpers.build_scoring_bundle(MODEL, "outputs/guide_bundle.zip")
info = helpers.describe_bundle(bundle)
print(f"Bundle: {info['bytes']:,} bytes   (model zip: {os.path.getsize(MODEL):,} bytes)\n")
for name, size in sorted(info["files"].items()):
    print(f"  {name:22} {size:>10,} bytes")

{k: info["settings"][k] for k in ["log_column", "asset_id_mapping", "timestamp_mapping",
                                   "feature_frequency", "rolling_window", "rolling_feature_type",
                                   "weights_source"]}

---
## 3. Deploy

`accept_storage` has no default. Without `True` the SDK refuses before anything is sent.
Pass `True` only once you (or the person you're doing this for) have read and accepted the
wording in section 1.

In [ ]:
dep = client.deploy_model(API_KEY, MODEL, DEPLOYMENT_NAME, accept_storage=True)
DEP_ID = dep["deployment_id"]
{k: dep[k] for k in ["deployment_id", "name", "asset_id_column", "timestamp_column",
                     "log_column", "feature_frequency", "lookback", "created_at"]}

**What the fields mean:**
- `asset_id_column`, `timestamp_column`, `log_column`: the columns new files must have, as in
  the training data. Other columns can be present; they are not sent.
- `feature_frequency`: the length of one scored period (one score per asset per period).
- `lookback`: how much earlier data to send in front of new logs so the rolling features
  are complete (section 5). For a `sum` or `flag` model it is twice the rolling window.

In [ ]:
pd.DataFrame(client.list_deployments(API_KEY))[
    ["deployment_id", "name", "feature_frequency", "score_calls", "last_scored_at", "created_at"]]

---
## 4. Score a one-off file

`client.score` sends only the three model columns and returns one row per asset and period,
with the same columns as `risk_scores.parquet` from Step 12. Nothing is stored.

In [ ]:
one_off = client.score(API_KEY, DEP_ID, str(week[5]))   # a DataFrame works too
one_off

In [ ]:
one_off.diagnostics

**Reading the diagnostics:**
- `rows_received` / `rows_in`: rows received, and rows left after dropping those without an
  asset ID or a readable timestamp.
- `rows_in` / `rows_matched` / `unmatched_share`: how many logs matched a subsystem. Some
  logs never match (routine entries the model has no subsystem for), so compare the share
  with what the model usually sees. A clear rise means the new logs use codes or wording
  the model never saw (see section 9).
- `assets_scored`, `periods_scored`, `period_min` / `period_max`: what came back.

In [ ]:
cols = ["asset_id", "period_start", "operational_risk"]
one_off.scores.sort_values("operational_risk", ascending=False)[cols].head(10)

---
## 5. Why history matters (`lookback` and `score_from`)

Each period's score is built from the logs in the **rolling window** before it (7 days
here). Score a week on its own and the first days of that week only see part of their
window, so they come out too low.

The fix is to send `lookback` of earlier logs in front of the new ones, and set `score_from`
to where the new data starts. The earlier logs feed the rolling window; only periods from
`score_from` come back.

Below, week 7 is scored both ways. **What you should see:** the two agree by the end of the
week, but early in the week the version without history is lower.

In [ ]:
new = pd.read_csv(week[7])
ts_col = dep["timestamp_column"]
start = pd.to_datetime(new[ts_col]).min()
lookback = pd.Timedelta(dep["lookback"])

earlier = pd.concat([pd.read_csv(TRAINING)] + [pd.read_csv(week[n]) for n in range(1, 7)])
history = earlier[pd.to_datetime(earlier[ts_col]) >= start - lookback]

alone = client.score(API_KEY, DEP_ID, new).scores
with_history = client.score(API_KEY, DEP_ID, pd.concat([history, new]), score_from=start).scores
print(f"Sent {len(history):,} rows of history ({lookback}) in front of {len(new):,} new rows.")

def daily_peak(s):
    return s.assign(day=pd.to_datetime(s["period_start"]).dt.date).groupby("day")["operational_risk"].max()

pd.DataFrame({"without history": daily_peak(alone), "with history": daily_peak(with_history)}).round(1)

Doing this by hand every time is error-prone: you have to keep the right history, avoid
sending the same rows twice, and replace the last period when more of it arrives. The
monitor does all of that for you.

---
## 6. Set up a monitor

A monitor is a **folder on your machine**:

```
guide_fleet_monitor/
├── monitor.json          settings: deployment, columns, alert bands (never the API key)
├── state.json            known assets, latest timestamp, number of runs
├── bundle.zip            the scoring bundle that was deployed
├── history/logs.parquet  the latest logs, kept for the rolling window (2 × lookback)
├── scores/scores.parquet every score so far, one row per asset and period
├── runs/<run_id>/run.json   what each run did
├── dashboard/index.html  rebuilt after every run
└── inbox/                drop new files here (section 8)
```

Create it from the command line:

```bash
amygda-risk init fleet_monitor --model complete_model_<id>.zip --name fleet-v1 \
    --accept-storage --history training_logs.csv
```

or from Python, as below. Either way it deploys the model (so `accept_storage` is needed
again), and:
- `history`: earlier logs, usually the training file, so the first run already has a full
  rolling window;
- `bands`: the watch and high levels agreed after Step 12. If you leave them out, they are
  set from the model's training scores (95th and 99th percentile). They then stay fixed so
  runs can be compared.

In [ ]:
mon = Monitor.init(
    MON_DIR,
    model          = MODEL,
    name           = MON_NAME,
    accept_storage = True,
    api_key        = API_KEY,
    fleet_name     = "Baggage sortation stations",
    history        = TRAINING,
    # bands        = {"watch": 35, "high": 45},   # ← the levels agreed after Step 12
)
print(json.dumps(mon.config, indent=1))

**`monitor.json` fields worth knowing:**
- `deployment`: the deployment it scores with, its columns and `lookback_seconds`.
- `bands`: the alert levels and how they were set (`basis`). Edit them here if needed.
- `date_format`: leave `null` to use the model's; set it if new files write dates differently.
- `api_key_env`: the environment variable the key is read from. The key itself is never saved.
- `dashboard_days`: how many recent days the dashboard shows.

---
## 7. Run it on weekly batches

Each run reads new files, drops rows it has already seen, adds the history the model needs,
scores, merges the scores into `scores/scores.parquet`, marks the newest period as
**provisional** (more of its logs may still arrive) and rebuilds the dashboard.

Feed it weeks 1 to 7, one at a time.

In [ ]:
for n in range(1, 8):
    r = mon.run(week[n])
    rec = r.record
    print(f"week {n:2}: {rec['rows_new']:4} new logs, {rec['periods_upserted']:4} periods scored, "
          f"unmatched {rec['unmatched_share']:.0%}, provisional {rec['provisional_periods']}")
    for w in r.warnings:
        print("   !", w)

**The provisional period.** The period holding the newest log is marked `provisional`: more
logs for it may still arrive, so its score only covers what has been received so far. The
next run scores it again in full and replaces it. (If the newest log matched no subsystem,
that period has no score and nothing is provisional.)

The sample week 6 ends at noon on Wednesday 14 February, part-way through a day. For a daily
or 12-hourly model that day (or half-day) is provisional until week 7 arrives.

In [ ]:
key = ["asset_id", "period_start"]
before = mon.scores
prov = before[before["provisional"]][key + ["operational_risk", "provisional", "run_id"]]
prov

In [ ]:
mon.run(week[8])
after = mon.scores.merge(prov[key], on=key)[key + ["operational_risk", "provisional", "run_id"]]
after   # the same period: no longer provisional, and written by the newer run

**Run the same file twice?** Nothing happens: every row has been seen, so there is nothing
to score and no API call is made.

In [ ]:
mon.run(week[8])

**Since the last run.** Each run record lists, per asset, where it stood before the run, where
it is now, the change, and the highest score in the new data with its band. The command line
prints the assets that reached watch or high; the dashboard shows the full table.

In [ ]:
pd.DataFrame(mon.runs()[-2]["changes"]).head(8)

**The dashboard** (`dashboard/index.html`) is one self-contained page. Open it in a browser:
- **Since the last run** and **Run health** at the top;
- the fleet heatmap (hatched cells are provisional) and the asset ranking;
- **Asset detail**: the charts open daily (the highest risk each day). Drag across the trend
  chart to zoom in; the resolution gets finer as you zoom, down to the model's own
  frequency. Double-click or press Esc to zoom out.

In [ ]:
dash = MON_DIR / "dashboard" / "index.html"
print(dash.resolve())
webbrowser.open(dash.resolve().as_uri())

---
## 8. Schedule it

For regular scoring, point your export at the monitor's `inbox/` folder and run
`amygda-risk run <folder>` on a schedule. With no `--input`, a run takes every CSV, XLSX or
parquet file in `inbox/`, oldest first, and moves them to `inbox/processed/<run_id>/`
when it's done. If the inbox is empty it does nothing.

Here, weeks 9–13 go into the inbox and one run picks them all up:

In [ ]:
for n in range(9, 14):
    shutil.copy(week[n], MON_DIR / "inbox" / week[n].name)
print("Waiting:", [p.name for p in mon.inbox_files()])

!{AMYGDA_RISK} run {MON_DIR}

print("Left in inbox:", [p.name for p in mon.inbox_files()])

**The schedule itself.** The scheduled job needs the API key in its environment as
`AMYGDA_API_KEY`. Keep it in a file only you can read, not in the job line.

**macOS and Linux (cron)** — every day at 06:00. Run `crontab -e` and add:
```bash
0 6 * * * . $HOME/.amygda_env && /path/to/venv/bin/amygda-risk run /path/to/fleet_monitor >> /path/to/fleet_monitor/cron.log 2>&1
```
where `$HOME/.amygda_env` holds `export AMYGDA_API_KEY=...` and has `chmod 600`.

**macOS (launchd)** — save as `~/Library/LaunchAgents/io.amygda.fleet-monitor.plist`, then
`launchctl load ~/Library/LaunchAgents/io.amygda.fleet-monitor.plist`:
```xml
<?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE plist PUBLIC "-//Apple//DTD PLIST 1.0//EN" "http://www.apple.com/DTDs/PropertyList-1.0.dtd">
<plist version="1.0"><dict>
  <key>Label</key><string>io.amygda.fleet-monitor</string>
  <key>ProgramArguments</key><array>
    <string>/bin/sh</string><string>-c</string>
    <string>. $HOME/.amygda_env &amp;&amp; /path/to/venv/bin/amygda-risk run /path/to/fleet_monitor</string>
  </array>
  <key>StartCalendarInterval</key><dict><key>Hour</key><integer>6</integer><key>Minute</key><integer>0</integer></dict>
  <key>StandardOutPath</key><string>/path/to/fleet_monitor/launchd.log</string>
  <key>StandardErrorPath</key><string>/path/to/fleet_monitor/launchd.log</string>
</dict></plist>
```

**Windows (Task Scheduler)** — set the key once for your user, then create a daily task:
```bat
setx AMYGDA_API_KEY "your-key"
schtasks /Create /SC DAILY /ST 06:00 /TN "Amygda fleet monitor" ^
  /TR "C:\path\to\venv\Scripts\amygda-risk.exe run C:\path\to\fleet_monitor"
```

Two runs can't overlap: a run holds `monitor.lock` and a second one stops with a message.
A lock left by a crashed run is cleared automatically.

---
## 9. Run health, and when to retrain

`amygda-risk status <folder>` (or `mon.status()`) summarises the monitor. The run records,
and the **Run health** panel on the dashboard, show how each run went.

In [ ]:
!{AMYGDA_RISK} status {MON_DIR}

In [ ]:
cols = ["run_id", "status", "rows_read", "rows_already_seen", "rows_new",
        "periods_upserted", "unmatched_share", "provisional_periods", "warnings"]
pd.DataFrame(mon.runs()).reindex(columns=cols)

**What to watch:**

| Signal | What it means | What to do |
|---|---|---|
| A warning that unmatched logs are "up from" the first run's share, or that almost none matched | New log codes or wording the model never saw, or the wrong log column | Check the export. If the logs really changed, retrain |
| "asset(s) not seen before" | New assets. They are scored, but their first `lookback` days have little history | Usually nothing; their scores settle after `lookback` |
| "rows are older than the history kept here" | A late or re-sent old file. Those rows are skipped so they can't replace good scores | Re-score that period one-off with `amygda-risk score` if you need it |
| A file "could not be read" with the date format | The export changed its date format | Set `date_format` in `monitor.json` |
| `no_new_data` | Everything in the files had been seen before | Nothing |

**When to retrain:** when the fleet or its logging changes (new equipment, new log codes, a
new maintenance system), when unmatched logs keep rising, when failures stop being preceded
by higher risk, or about once a year. Retrain through `01_labelling_guide.ipynb` and
`02_risk_score_guide.ipynb`, deploy the new model under a new name (e.g. `fleet-v2`) and
create a new monitor with the old monitor's `history/logs.parquet` as `history`.

**Rows are matched on their exact values.** A re-export that rounds or reformats timestamps
counts as new rows. Keep exports consistent, or send only new rows.

---
## 10. Manage and delete

- `client.list_deployments(API_KEY)` lists your organisation's deployments, with
  `score_calls` and `last_scored_at`.
- `client.delete_deployment(API_KEY, deployment_id)` deletes a deployment and its stored
  bundle immediately. A monitor using it stops working; delete or re-create the monitor.
- A deployment unused for **90 days** is deleted automatically.
- Deleting a monitor folder only deletes local files. Delete its deployment as well.

`amygda-risk score <folder> --input file.csv` scores a file one-off with the monitor's model,
and writes its scores and a dashboard to a separate folder without changing the monitor:

In [ ]:
!{AMYGDA_RISK} score {MON_DIR} --input {week[3]} --output outputs/guide_oneoff

Clean up what this guide created. Skip this cell to keep the monitor.

In [ ]:
for d in client.list_deployments(API_KEY):
    if d["name"] in (DEPLOYMENT_NAME, MON_NAME):
        client.delete_deployment(API_KEY, d["deployment_id"])
shutil.rmtree(MON_DIR, ignore_errors=True)
shutil.rmtree("outputs/guide_oneoff", ignore_errors=True)
client.list_deployments(API_KEY)